# Raio-X do Mercado de Trabalho Industrial de SC — pipeline completo

**Datathon CDIA 2026 · UniSENAI.** Site: https://datathon-ebon.vercel.app · Contrato do projeto: `docs/SDD_v2.md`.

**Pergunta central:** quais regiões e setores industriais de Santa Catarina estão enfrentando apagão de mão de obra ou expansão acelerada, e como os indicadores de renda e escolaridade explicam esses gargalos?

**Resposta:** o apagão da indústria catarinense aparece como **rotatividade**, não como falta de gente. Metade dos desligamentos é a pedido do trabalhador, o saldo quase não cresce, quem entra tem a mesma escolaridade de quem sai e, onde a mesma ocupação paga mais que no resto do estado, menos gente pede para sair.

Este notebook **só chama as funções de `src/`** e explica cada etapa, da carga dos microdados até a classificação por IA, o modelo estatístico e a exportação para o site. Ele roda de cima a baixo (*Restart & Run All*); o mesmo pipeline roda por linha de comando com `python run_pipeline.py`.

| Etapa | Módulo | Saída |
| --- | --- | --- |
| 1. Ingestão | `src/ingest.py` | `data/interim/caged_raw.parquet` |
| 2. Referências (IBGE e dicionário) | `src/referencias.py` | `data/reference/` |
| 3. Limpeza, filtros e junções | `src/clean.py` | `data/interim/caged_ind.parquet` |
| 4. Validação com o boletim da FIESC | `src/testes.py` | `reports/testes.md` |
| 5. IA generativa: macrofamílias de ocupação | `src/macrofamilias.py` | `data/reference/macrofamilias.csv` |
| 6. KPIs | `src/kpis.py` | `data/processed/` |
| 7. Testes de robustez T2–T6 | `src/testes.py` | `reports/testes.md` |
| 8. Modelo estatístico das saídas a pedido | `src/modelo.py` | `reports/modelo_saida_pedido.md` |
| 9. Figuras e recomendações | `src/figuras.py`, `src/recomendacoes.py` | `reports/figures/`, `app/recomendacoes.yaml` |
| 10. Exportação para o site e o assistente | `src/exportar_web.py`, `src/base_conhecimento.py`, `src/dados_agente.py` | `web/public/`, `web/api/` |

**Antes de rodar:** coloque os 3 arquivos do evento em `data/raw/` (veja o README).

In [1]:
import json, os, sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))
import pandas as pd
import polars as pl
import yaml
from IPython.display import Markdown, display
import config as cfg
from src import (base_conhecimento, clean, dados_agente, exportar_web, figuras, ingest, kpis,
                 macrofamilias, modelo, recomendacoes, referencias, testes)
from src.log import iniciar_log
pl.Config.set_tbl_rows(25); pl.Config.set_tbl_width_chars(180); pl.Config.set_fmt_str_lengths(60)
OFFLINE = cfg.MUNICIPIOS_REGIOES_CSV.exists() or os.getenv("DATATHON_OFFLINE") == "1"
print("Raiz:", RAIZ.name, "· regiões do IBGE já baixadas:", OFFLINE)

Raiz: datathon · regiões do IBGE já baixadas: True


## 1. Ingestão (regras R1–R2)

O CSV do Novo CAGED (413 MB, 3,4 milhões de linhas) é lido **uma vez** com Polars, com **todas as colunas como texto**: códigos de CBO, CNAE e município nunca viram número (senão a CBO `071105` perderia o zero e a CNAE `01` viraria `1`). Os nomes das colunas viram ASCII, salário e horas trocam vírgula por ponto, e o resultado vai para `data/interim/caged_raw.parquet`.

In [2]:
iniciar_log()
raw = ingest.main()
print(f"{raw.height:,} linhas × {raw.width} colunas")
raw.head(3)


## Ingestão (R1–R2)

| Item | Valor |
| --- | --- |
| Linhas lidas | 3,397,416 |
| Colunas | 28 |
| Nulos em idade | 2,202 |
| Salário não numérico | 0 |
| Tempo de leitura | 6.3 s |
| Pico de memória do processo | 1,001 MB |

3,397,416 linhas × 28 colunas


competenciamov,regiao,uf,municipio,secao,subclasse,saldomovimentacao,cbo2002ocupacao,categoria,graudeinstrucao,idade,horascontratuais,racacor,sexo,tipoempregador,tipoestabelecimento,tipomovimentacao,tipodedeficiencia,indtrabintermitente,indtrabparcial,salario,tamestabjan,indicadoraprendiz,origemdainformacao,competenciadec,indicadordeforadoprazo,unidadesalariocodigo,valorsalariofixo
str,str,str,str,str,str,i8,str,str,str,i16,f64,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,f64
"""202509""","""4""","""42""","""420890""","""F""","""4120400""",1,"""715505""","""101""","""7""",40,44.0,"""1""","""1""","""0""","""1""","""97""","""0""","""0""","""0""",2018.0,"""5""","""0""","""1""","""202509""","""0""","""5""",2018.0
"""202509""","""4""","""42""","""421605""","""G""","""4671100""",-1,"""632120""","""101""","""7""",47,44.0,"""1""","""1""","""0""","""1""","""40""","""0""","""0""","""0""",1693.35,"""5""","""0""","""1""","""202509""","""0""","""5""",1693.35
"""202509""","""4""","""42""","""420820""","""F""","""4120400""",1,"""715505""","""101""","""6""",30,44.0,"""1""","""1""","""0""","""1""","""97""","""0""","""0""","""0""",2556.0,"""5""","""0""","""1""","""202509""","""0""","""5""",2556.0


## 2. Referências (R11, R12, R12b)

- **Municípios do evento:** só as linhas com código IBGE `42xxxxx` (o arquivo tem rodapé HTML). A chave com o CAGED é o código de 6 dígitos.
- **Microrregiões e mesorregiões:** API de localidades do IBGE; o mapa vem da API de malhas (GeoJSON), com os anéis reorientados para o padrão do D3/Plotly.
- **Dicionário:** títulos das CBOs e descrições das subclasses CNAE.

In [3]:
referencias.main(offline=OFFLINE)
reg = pl.read_csv(cfg.MUNICIPIOS_REGIOES_CSV, infer_schema_length=0)
print(f"{reg['cod7'].n_unique()} municípios · {reg['cod_microrregiao'].n_unique()} microrregiões · {reg['cod_mesorregiao'].n_unique()} mesorregiões")


## Dicionário (R12b)

| Item | Valor |
| --- | --- |
| Títulos de CBO | 2,778 |
| Subclasses CNAE | 1,357 |


## Municípios do evento (R11)

| Item | Valor |
| --- | --- |
| Municípios válidos | 295 |
| População total (Censo 2022) | 7,610,361 |

Modo offline: regiões e GeoJSON do IBGE não foram baixados.
295 municípios · 20 microrregiões · 6 mesorregiões


## 3. Limpeza, filtros e junções (R3–R14)

Janela set/2025–ago/2026; indústria = seções CNAE B–F (análise principal na transformação, C); base de salário só com celetistas comuns entre R$ 1.000 e R$ 30.000; escolaridade em faixas (o código 80, pós-graduação, nunca é tratado como "maior" que doutorado); grupos setoriais no padrão do boletim da FIESC; junção com população (Censo 2022) e regiões. **Linhas idênticas são mantidas**, como nos números oficiais. Cada regra registra quantas linhas entraram e saíram.

In [4]:
ind = clean.main(sem_regioes=not cfg.MUNICIPIOS_REGIOES_CSV.exists())
del raw
log = open(cfg.LOG_PIPELINE, encoding="utf-8").read()
display(Markdown(log[log.index("## Limpeza"):]))


## Limpeza e junções (R3–R14)

| Item | Valor |
| --- | --- |
| Entrada (caged_raw.parquet) | 3,397,416 |
| R4 janela 202509–202608 | 3,397,416 → 3,391,288 (−6,128) |
| R5 seções BCDEF | 3,391,288 → 1,139,633 (−2,251,655) |
| R6 exclusões | coluna ausente: nada removido (V2) |
| R7 cópias excedentes de linhas idênticas mantidas (B–F) | 26,916 |
| R7 fora do prazo mantidas | 9,599 |
| R11 municípios casados | 295 de 295; sem população: 0 |
| R12 regiões | 20 microrregiões; nulos: 0 |
| R12b CBO sem título | 131 linhas |
| R12b subclasse sem descrição | 0 linhas |
| Nulos em divisao_cnae | 0 |
| Nulos em cbo2002ocupacao | 0 |
| Nulos em grupo_setorial | 0 |
| Nulos em microrregiao | 0 |
| Saída (caged_ind.parquet) | 1,139,633 linhas, 41 colunas |



## Limpeza e junções (R3–R14)

| Item | Valor |
| --- | --- |
| Entrada (caged_raw.parquet) | 3,397,416 |
| R4 janela 202509–202608 | 3,397,416 → 3,391,288 (−6,128) |
| R5 seções BCDEF | 3,391,288 → 1,139,633 (−2,251,655) |
| R6 exclusões | coluna ausente: nada removido (V2) |
| R7 cópias excedentes de linhas idênticas mantidas (B–F) | 26,916 |
| R7 fora do prazo mantidas | 9,599 |
| R11 municípios casados | 295 de 295; sem população: 0 |
| R12 regiões | 20 microrregiões; nulos: 0 |
| R12b CBO sem título | 131 linhas |
| R12b subclasse sem descrição | 0 linhas |
| Nulos em divisao_cnae | 0 |
| Nulos em cbo2002ocupacao | 0 |
| Nulos em grupo_setorial | 0 |
| Nulos em microrregiao | 0 |
| Saída (caged_ind.parquet) | 1,139,633 linhas, 41 colunas |


## 4. Validação com o número oficial (T1) e conferência dos fatos de referência

Antes de qualquer análise, o saldo jan–mai/2026 calculado aqui é comparado com o boletim de emprego industrial da FIESC. Na transformação a diferença é de **0,01%**. Na construção fica 0,504%, pouco acima da tolerância de 0,5%: a equipe decidiu declarar a diferença em vez de mexer nos filtros.

In [5]:
cfg.TESTES_MD.unlink(missing_ok=True)  # recomeça o registro dos testes
testes.t1_reconciliacao(ind)

| 2026-11-12 | T1 | Transformação (C) | 24,271 | 24,269 | diferença ≤ 0,5% | aprovado (0.01%) |
| 2026-11-12 | T1 | Indústria geral (B–E) | 24,536 | 24,527 | diferença ≤ 0,5% | aprovado (0.04%) |
| 2026-11-12 | T1 | Construção (F) | 10,775 | 10,721 | diferença ≤ 0,5% | REPROVADO (0.50%) |
| 2026-11-12 | T1 | Indústria total (B–F) | 35,311 | 35,248 | diferença ≤ 0,5% | aprovado (0.18%) |


False

## 5. IA generativa: macrofamílias de ocupação (Google Gemini)

São 1.578 ocupações (CBOs) na indústria; as **354** que cobrem 95% das movimentações são classificadas pelo Gemini em **15 áreas × 4 níveis de formação** (Qualificação até 200h, acima de 200h, Técnico, Superior), no formato do Mapa do Trabalho Industrial (SENAI/CNI).

Como evitamos invenção:
- **Lista fechada:** o esquema de saída (Pydantic → JSON Schema) só aceita os valores exatos das 15 áreas e 4 níveis; temperatura 0; lotes de 50.
- **Conferência do lote:** o conjunto de CBOs devolvido precisa ser idêntico ao enviado, senão o lote é refeito.
- **Validação:** M1 (confiança baixa → revisão), M2 (nível incoerente com o grande grupo da CBO → revisão), M3 (amostra para dois revisores humanos), M4 (herança pela família da CBO para as ocupações pequenas).

Por padrão (`RODAR_LLM = False` em `config.py`) o notebook **não chama a API**: lê o cache das respostas e o `macrofamilias.csv` versionado. Para refazer: `python src/macrofamilias.py --llm` com a `GEMINI_API_KEY` no `.env`.

In [6]:
print(macrofamilias.texto_prompt()[:1800], "\n[...]")

Você é um especialista em classificação de ocupações (CBO 2002) e em formação profissional industrial
no Brasil, no formato do Mapa do Trabalho Industrial (SENAI/CNI).

Classifique cada ocupação da lista em UMA área e UM nível de formação, usando SOMENTE os valores abaixo.
Nunca crie categorias, nunca altere nem invente códigos de CBO. Devolva exatamente as CBOs recebidas, uma vez cada.

ÁREAS:
- Alimentos e bebidas: Ocupações específicas do processamento de alimentos e bebidas: abate, desossa, laticínios, panificação industrial
- Têxtil, confecção e calçados: Fiação, tecelagem, corte, costura, acabamento, calçados
- Metalmecânica e soldagem: Usinagem, caldeiraria, soldagem, fundição, estamparia, montagem mecânica especializada
- Manutenção industrial: Mecânicos, eletricistas e eletromecânicos de manutenção, lubrificadores
- Eletroeletrônica e automação: Montagem e operação eletroeletrônica, instrumentação, automação, bobinagem
- Madeira e móveis: Serraria, marcenaria, montagem de móve

Código da chamada (resumo de `src/macrofamilias.py`):

```python
client = genai.Client()  # GEMINI_API_KEY vem do .env, nunca do código
config = types.GenerateContentConfig(system_instruction=texto_prompt(), temperature=0,
    response_mime_type="application/json", response_schema=list[Classificacao])
resp = client.models.generate_content(model=cfg.GEMINI_MODEL, contents=linhas_entrada(lote), config=config)
```

In [7]:
macrofamilias.main()
mf = pl.read_csv(cfg.MACROFAMILIAS_CSV, infer_schema_length=0)
print(mf["origem"].value_counts().sort("count", descending=True))
mov = pl.read_parquet(cfg.CAGED_IND_PARQUET, columns=["cbo2002ocupacao"]).group_by("cbo2002ocupacao").len("movimentos")
(mf.join(mov, left_on="cbo", right_on="cbo2002ocupacao").sort("movimentos", descending=True)
   .select("cbo", "titulo_cbo", "area", "nivel", "confianca", "movimentos").head(15))

CBOs de B–F: 1578 · no escopo (95% dos movimentos): 354 · sem título no dicionário: 9 (0 no escopo)


# Validação das macrofamílias

Execução: 2026-11-12 22:01

- CBOs de B–F: 1578 · com área e nível: 1578 · fora da lista fechada: 0
- Origem: llm 354, outros 751, heranca 473
- M1 (confiança baixa): 1 · M2 (divergência com o grande grupo): 7

## Por área

| area | CBOs | Movimentos | % |
| --- | --- | --- | --- |
| Operação de linha e montagem | 24 | 237,592 | 20.8 |
| Construção | 101 | 224,123 | 19.7 |
| Têxtil, confecção e calçados | 96 | 121,112 | 10.6 |
| Gestão, administrativo e comercial | 97 | 103,213 | 9.1 |
| Logística e movimentação | 42 | 91,291 | 8.0 |
| Metalmecânica e soldagem | 106 | 90,326 | 7.9 |
| Alimentos e bebidas | 54 | 63,997 | 5.6 |
| Serviços de apoio | 34 | 43,011 | 3.8 |
| Manutenção industrial | 32 | 28,129 | 2.5 |
| Outros | 751 | 27,796 | 2.4 |
| Madeira e móveis | 52 | 25,674 | 2.3 |
| Plástico, borracha e química | 57 | 23,807 | 2.1 |
| Eletroeletrônica e automação | 48 | 22,755 | 2.0 |
| Qualidade e controle | 16 | 17,572 | 1.5 |
| Cerâmica, vidro e min

cbo,titulo_cbo,area,nivel,confianca,movimentos
str,str,str,str,str,u32
"""784205""","""Alimentador de Linha de Producao""","""Operação de linha e montagem""","""Qualificação até 200h""","""alta""",207461
"""717020""","""Servente de Obras""","""Construção""","""Qualificação até 200h""","""alta""",68428
"""715210""","""Pedreiro""","""Construção""","""Qualificação acima de 200h""","""alta""",43592
"""848520""","""Magarefe""","""Alimentos e bebidas""","""Qualificação acima de 200h""","""alta""",36298
"""411005""","""Auxiliar de Escritorio, em Geral""","""Gestão, administrativo e comercial""","""Qualificação acima de 200h""","""alta""",26198
"""411010""","""Assistente Administrativo""","""Gestão, administrativo e comercial""","""Qualificação acima de 200h""","""alta""",23067
"""763210""","""Costureiro na Confeccao em Serie""","""Têxtil, confecção e calçados""","""Qualificação acima de 200h""","""alta""",20199
"""763215""","""Costureiro, a Maquina na Confeccao em Serie""","""Têxtil, confecção e calçados""","""Qualificação acima de 200h""","""alta""",19137
"""414105""","""Almoxarife""","""Logística e movimentação""","""Qualificação acima de 200h""","""alta""",18757


## 6. Indicadores (KPIs)

| KPI | Pergunta | Fórmula |
| --- | --- | --- |
| 1. % de saídas a pedido | O trabalhador sai por conta própria? | desligamentos a pedido ÷ desligamentos (adaptação da taxa de *quits* do JOLTS/BLS) |
| 2. Saldo por 10 mil hab. | Onde há expansão? | (admissões − desligamentos) ÷ população (Censo 2022) × 10.000 |
| 3. Prêmio de contratação | A empresa paga mais para repor? | mediana do salário de admissão ÷ de desligamento, na mesma CBO × microrregião, ponderado |
| 4. Descasamento de escolaridade | Quem entra estudou menos? | % superior nos admitidos − nos desligados, na mesma CBO × microrregião, ponderado |

Os KPIs 3 e 4 são calculados **dentro da mesma ocupação e região** para não confundir efeito de setor com mistura de cargos.

In [8]:
saidas = kpis.main()
(pl.read_csv(cfg.PROCESSED / "kpis_grupo_setorial.csv").filter(pl.col("segmento") == "Transformação")
   .select("grupo_setorial", "movimentos", "saldo", pl.col("kpi1_pct_a_pedido").round(1), pl.col("kpi2_saldo_10mil_hab").round(1),
           pl.col("kpi3_premio_contratacao").round(3)).sort("kpi1_pct_a_pedido", descending=True))


## KPIs (seção 5)

| Item | Valor |
| --- | --- |
| kpis_segmento | 3 linhas |
| kpis_grupo_setorial | 13 linhas |
| kpis_divisao | 37 linhas |
| kpis_segmento_sem_cbo_dominante | 3 linhas |
| kpis_grupo_setorial_sem_cbo_dominante | 13 linhas |
| perfil_escolaridade_grupo | 104 linhas |
| perfil_salario_grupo | 26 linhas |
| kpis_microrregiao | 60 linhas |
| kpis_micro_grupo | 257 linhas |
| quadrantes | 169 linhas |
| perfil_escolaridade_micro_grupo | 1962 linhas |
| perfil_salario_micro_grupo | 512 linhas |
| quociente_locacional | 257 linhas |
| Polos (QL > 1,1 / 1,25 / 1,5 com ≥1.000 mov.) | 62 / 54 / 41 |
| kpis_macrofamilia | 115 linhas |
| kpis_grupo_macrofamilia | 495 linhas |
| kpis_micro_macrofamilia | 1760 linhas |
| kpis_micro_grupo_macrofamilia | 5540 linhas |
| Tamanho de data/processed | 2.20 MB (limite 20 MB) |



grupo_setorial,movimentos,saldo,kpi1_pct_a_pedido,kpi2_saldo_10mil_hab,kpi3_premio_contratacao
str,i64,i64,f64,f64,f64
"""Alimentos e bebidas""",182711,4731,56.2,6.2,0.983
"""Madeira e móveis""",78864,-2710,53.1,-3.6,0.975
"""Têxtil, confecção e calçados""",195178,-5916,52.4,-7.8,0.971
"""Eletroeletrônica""",42338,1646,50.1,2.2,0.971
"""Plástico e borracha""",65547,265,49.7,0.3,0.983
"""Papel, celulose e gráfica""",27488,340,48.3,0.4,0.984
"""Minerais não metálicos e cerâmica""",33492,-842,47.4,-1.1,0.982
"""Química e farmacêutica""",16391,341,46.1,0.4,0.976
"""Metalmecânica""",178307,3459,45.6,4.5,0.978


## 7. Testes de robustez (T2 a T6)

- **T2** separa o efeito de setor do efeito da mistura de ocupações (padronização indireta).
- **T3** mede quanto o resultado depende de uma única ocupação, o alimentador de linha (CBO 784205). **Reprovado:** por isso o site mostra os resultados com e sem ela.
- **T4** estabilidade entre semestres · **T5** dependência da definição de indústria · **T6** concordância da IA com revisores humanos.

In [9]:
testes.main_fase3()
display(Markdown(open(cfg.TESTES_MD, encoding="utf-8").read()))

| 2026-11-12 | T2a | C, mesma CBO em ≥3 divisões | 15 CBOs; amplitude mediana 11.1 p.p.; 15 acima de 5 p.p. | ≥ 10 CBOs | informativo (reportar se < 10 CBOs) | aprovado |


| 2026-11-12 | T2b | C, 14 maiores divisões | amplitude padronizada 5.6 p.p. (observada 13.3) | > 5 p.p. | diferença entre maior e menor taxa padronizada | aprovado |
shape: (14, 5)
┌──────────────┬───────────┬───────────────────────┬─────────────┬───────────────────────────────────┐
│ divisao_cnae ┆ observado ┆ esperado_pela_mistura ┆ padronizado ┆ grupo_setorial                    │
│ ---          ┆ ---       ┆ ---                   ┆ ---         ┆ ---                               │
│ str          ┆ f64       ┆ f64                   ┆ f64         ┆ str                               │
╞══════════════╪═══════════╪═══════════════════════╪═════════════╪═══════════════════════════════════╡
│ 29           ┆ 52.7      ┆ 50.1                  ┆ 53.5        ┆ Metalmecânica                     │
│ 16           ┆ 54.2      ┆ 51.9                  ┆ 53.3        ┆ Madeira e móveis                  │
│ 10           ┆ 56.5      ┆ 54.6                  ┆ 52.9        ┆ Alimentos e bebidas           

| 2026-11-12 | T3 | KPI 1, 5 piores divisões sem CBO 784205 | ordem com: 10, 14, 16, 21, 27; maior mudança 4 | ≤ 1 posição | ranking estável | REPROVADO |


| 2026-11-12 | T3 | KPI 2, 5 piores divisões sem CBO 784205 | ordem com: 14, 16, 31, 23, 13; maior mudança 3 | ≤ 1 posição | ranking estável | REPROVADO |


| 2026-11-12 | T4 | C, microrregião × grupo setorial | r = 0.80 em 111 recortes | > 0,7 | estabilidade entre semestres | aprovado |


| 2026-11-12 | T5 | KPI 1 do estado, C × B–E | 50.91% × 50.63% (dif. 0.28 p.p.) | < 1 p.p. |  | aprovado |


| 2026-11-12 | T5 | Top 5 microrregiões por KPI 1, C × B–E | C: São Miguel do Oeste, Concórdia, Ituporanga, Tabuleiro, Rio do Sul | B–E: São Miguel do Oeste, Concórdia, Ituporanga, Tabuleiro, Rio do Sul; 0 troca(s) | ≤ 1 troca |  | aprovado |
| 2026-11-12 | T6 | Amostra M3 | 0 de 30 linhas preenchidas pelos dois revisores | ≥ 85% | concordância LLM × revisores | pendente |


# Testes de validação

| Data | Teste | Grupo | Obtido | Referência | Critério | Resultado |
| --- | --- | --- | --- | --- | --- | --- |
| 2026-11-12 | T1 | Transformação (C) | 24,271 | 24,269 | diferença ≤ 0,5% | aprovado (0.01%) |
| 2026-11-12 | T1 | Indústria geral (B–E) | 24,536 | 24,527 | diferença ≤ 0,5% | aprovado (0.04%) |
| 2026-11-12 | T1 | Construção (F) | 10,775 | 10,721 | diferença ≤ 0,5% | REPROVADO (0.50%) |
| 2026-11-12 | T1 | Indústria total (B–F) | 35,311 | 35,248 | diferença ≤ 0,5% | aprovado (0.18%) |
| 2026-11-12 | T2a | C, mesma CBO em ≥3 divisões | 15 CBOs; amplitude mediana 11.1 p.p.; 15 acima de 5 p.p. | ≥ 10 CBOs | informativo (reportar se < 10 CBOs) | aprovado |
| 2026-11-12 | T2b | C, 14 maiores divisões | amplitude padronizada 5.6 p.p. (observada 13.3) | > 5 p.p. | diferença entre maior e menor taxa padronizada | aprovado |
| 2026-11-12 | T3 | KPI 1, 5 piores divisões sem CBO 784205 | ordem com: 10, 14, 16, 21, 27; maior mudança 4 | ≤ 1 posição | ranking estável | REPROVADO |
| 2026-11-12 | T3 | KPI 2, 5 piores divisões sem CBO 784205 | ordem com: 14, 16, 31, 23, 13; maior mudança 3 | ≤ 1 posição | ranking estável | REPROVADO |
| 2026-11-12 | T4 | C, microrregião × grupo setorial | r = 0.80 em 111 recortes | > 0,7 | estabilidade entre semestres | aprovado |
| 2026-11-12 | T5 | KPI 1 do estado, C × B–E | 50.91% × 50.63% (dif. 0.28 p.p.) | < 1 p.p. |  | aprovado |
| 2026-11-12 | T5 | Top 5 microrregiões por KPI 1, C × B–E | C: São Miguel do Oeste, Concórdia, Ituporanga, Tabuleiro, Rio do Sul | B–E: São Miguel do Oeste, Concórdia, Ituporanga, Tabuleiro, Rio do Sul; 0 troca(s) | ≤ 1 troca |  | aprovado |
| 2026-11-12 | T6 | Amostra M3 | 0 de 30 linhas preenchidas pelos dois revisores | ≥ 85% | concordância LLM × revisores | pendente |


## 8. Modelo estatístico: o que está associado às saídas a pedido

Dois níveis, porque cada um responde uma pergunta diferente.

**Nível 1 — Mercado: pagar acima do mercado retém?** A unidade é a célula ocupação (CBO) × microrregião, com pelo menos 100 desligamentos (620 células, 75% dos desligamentos). Logit fracionário com efeitos fixos:

$$\operatorname{logit}(p_c)=\ln\frac{p_c}{1-p_c}=\beta_0+\beta_1\ln s_c+\beta_2\ln \pi_c+\beta_3 g_c+\sum_{k=4}^{7}\beta_k z_{kc}+\gamma_{\text{setor}}+\delta_{\text{meso}}+\theta_{\text{ocup}}$$

- $p_c$: parcela das saídas a pedido na célula; $s_c$: mediana salarial da célula ÷ mediana da mesma ocupação em SC; $\pi_c$: prêmio de contratação; $g_c$: crescimento; $z_{kc}$: composição (% superior, jovens, mulheres, empresas 500+).
- Estimação por quase-verossimilhança binomial (GLM, ligação logit), ponderada pelos desligamentos, erros-padrão robustos (HC1).

**Do coeficiente aos pontos percentuais:** a razão de chances de +10% de salário é $1{,}1^{\hat\beta_1}$; pela derivada do logit, $\partial p/\partial \ln s=\beta_1 p(1-p)$, então $\Delta p \approx \hat\beta_1 \ln(1{,}1)\,\bar p(1-\bar p)$; o valor exato é o efeito marginal médio

$$\Delta p=\frac{1}{N}\sum_{c=1}^{N}\Big[\Lambda\big(x_c'\hat\beta+\hat\beta_1\ln 1{,}1\big)-\Lambda\big(x_c'\hat\beta\big)\Big],\qquad \Lambda(z)=\frac{1}{1+e^{-z}}.$$

**Nível 2 — Perfil: entre quem saiu, quem saiu a pedido?** Regressão logística individual, $P(y_i=1\mid x_i)=\Lambda(\alpha+\sum_k\beta_k x_{ik}+\text{controles})$, validada fora da amostra contra um gradient boosting. Entre quem saiu, quem ganha bem acima da mediana sai mais por demissão da empresa (viés de seleção): por isso o efeito do salário é lido no Nível 1.

Associação, não causa. O CAGED não traz tempo de casa.

In [10]:
resumo = modelo.main()
m = resumo["mercado"]; d = m["grafico"]["derivacao"]
print(f"Nível 1: {m['celulas']} células · β1 = {d['beta1']:.3f} (IC 95% {d['ic_inf']:.3f} a {d['ic_sup']:.3f}; z = {d['z']:.2f})")
print(f"  razão de chances (+10%): {1.1 ** d['beta1']:.3f}")
print(f"  aproximação: {d['beta1']:.3f} × {d['ln11']:.4f} × {d['p_bar']:.3f} × {1 - d['p_bar']:.3f} = {d['aprox_pp']:.2f} p.p.")
print(f"  efeito marginal médio (exato): {d['exato_pp']:.2f} p.p. · deviance explicada: {100 * d['dev_explicada']:.0f}%")
print(f"Nível 2: {resumo['n']:,} desligamentos · AUC fora da amostra: logística {resumo['auc_logit']:.3f} × boosting {resumo['auc_gbm']:.3f}")
pd.DataFrame(m["robustez"])[["recorte", "celulas", "efeito_sal10_pp", "p_sal", "p_premio"]].round(4)


## Modelo das saídas a pedido

| Item | Valor |
| --- | --- |
| Desligamentos na amostra | 388,489 |
| Taxa a pedido na amostra | 52.0% |
| AUC fora da amostra (logística × boosting) | 0.616 × 0.646 |
| Pseudo-R² (McFadden) | 0.030 |
| Fator de maior amplitude | faixa_sal (10.0 p.p.) |
| Mercado: células ocupação × microrregião | 620 (313,848 desligamentos) |
| Mercado: +10% de salário relativo | -2.24 p.p. de saídas a pedido (p = 0.0002) |
| Tempo | 145 s |

Nível 1: 620 células · β1 = -0.963 (IC 95% -1.468 a -0.459; z = -3.74)
  razão de chances (+10%): 0.912
  aproximação: -0.963 × 0.0953 × 0.498 × 0.502 = -2.30 p.p.
  efeito marginal médio (exato): -2.24 p.p. · deviance explicada: 49%
Nível 2: 388,489 desligamentos · AUC fora da amostra: logística 0.616 × boosting 0.646


,recorte,celulas,efeito_sal10_pp,p_sal,p_premio
0,células com ≥ 50 desligamentos,1069,-1.7753,0.0003,0.6970
1,células com ≥ 200 desligamentos,340,-2.4965,0.0012,0.8040
2,sem o alimentador de linha (CBO 784205),601,-3.1434,0.0000,0.8074


In [11]:
(pd.read_csv(cfg.PROCESSED / "modelo_pedido_efeitos.csv")
   [["fator_nome", "nivel", "prob_prevista", "dif_pp_vs_ref", "or", "or_ic_inf", "or_ic_sup", "n"]].round(3))

,fator_nome,nivel,prob_prevista,dif_pp_vs_ref,or,or_ic_inf,or_ic_sup,n
0,Salário em relação à mediana da ocupação na re...,20% ou mais abaixo,51.731,-2.128,0.915,0.866,0.968,23601
1,Salário em relação à mediana da ocupação na re...,10% a 20% abaixo,54.118,0.259,1.011,0.960,1.065,38475
2,Salário em relação à mediana da ocupação na re...,3% a 10% abaixo,54.646,0.787,1.033,0.985,1.085,65110
3,Salário em relação à mediana da ocupação na re...,na mediana (±3%),53.859,0.000,NaN,NaN,NaN,93292
4,Salário em relação à mediana da ocupação na re...,3% a 10% acima,54.021,0.162,1.007,0.945,1.072,56745
5,Salário em relação à mediana da ocupação na re...,10% a 20% acima,51.370,-2.489,0.902,0.850,0.957,43595
6,Salário em relação à mediana da ocupação na re...,20% ou mais acima,44.601,-9.258,0.681,0.641,0.723,67671
7,Idade,até 17,55.395,3.909,1.178,1.093,1.268,5022
8,Idade,18 a 24,54.395,2.909,1.129,1.102,1.157,110110
9,Idade,25 a 29,52.407,0.921,1.039,1.015,1.064,74907


## 9. Figuras e recomendações

As figuras em PNG vão para `reports/figures/` (slides). As recomendações foram geradas como rascunho a partir dos quadrantes e reescritas e **aprovadas pela equipe** em `app/recomendacoes.yaml`; o pipeline não sobrescreve o arquivo aprovado. Cada uma liga uma **macrofamília da IA** a uma região e traz uma linha de base para acompanhar no CAGED.

In [12]:
figuras.main()
recomendacoes.main()
recs = yaml.safe_load(open(cfg.RECOMENDACOES_YAML, encoding="utf-8"))["recomendacoes"]
for i, r in enumerate(recs, 1):
    print(f"{i}. {r['acao']} — {r['regiao']}\n   Para quem: {r['macrofamilia']}\n   Linha de base: {r['indicador_de_acompanhamento']['linha_de_base']} {r['indicador_de_acompanhamento'].get('unidade', '')}\n")

PNG: reports\figures\01_pedido_por_grupo_setorial.png


PNG: reports\figures\02_t3_com_sem_alimentador.png


PNG: reports\figures\03_quadrantes.png


PNG: reports\figures\04_mapa_microrregioes.png


PNG: reports\figures\05_mapa_saldo_microrregioes.png
recomendacoes.yaml está aprovado pela equipe: não sobrescrito (candidatas só em recomendacoes_candidatas.md).
115 candidatas; 3 em nul como rascunho.
1. Programa de retenção na agroindústria — Oeste Catarinense (Chapecó, Joaçaba, Concórdia, Xanxerê, São Miguel do Oeste)
   Para quem: Alimentos e bebidas × Qualificação acima de 200h (e Operação de linha × até 200h)
   Linha de base: 64.6 %

2. Revisão de faixa salarial e programa de retenção no têxtil — Rio do Sul (Vale do Itajaí)
   Para quem: Têxtil, confecção e calçados × Qualificação acima de 200h
   Linha de base: 68.6 %

3. Requalificação do têxtil para eletroeletrônica e automação — Blumenau (Vale do Itajaí)
   Para quem: Têxtil, confecção e calçados × Qualificação acima de 200h → Eletroeletrônica e automação × Qualificação acima de 200h
   Linha de base: 189 vagas



## 10. Exportação para o site e para o assistente "Perguntar aos dados"

O site (React, em `web/`) **não calcula nada**: lê `web/public/dados.json`, gerado aqui. O assistente de IA do site (função `web/api/perguntar.js`, Gemini) responde dúvidas com uma arquitetura em camadas:

1. **Triagem sem IA:** saudação, "quem é você", tentativas de manipulação e perguntas prontas (inclusive variações de redação) respondem na hora.
2. **Busca por entidades:** identifica microrregiões, 295 municípios, setores e ocupações na pergunta e entrega ao modelo só as linhas exatas desses recortes (`web/api/_dados.js`), além da visão geral (`web/api/_base.js`).
3. **Resposta estruturada** (JSON): categoria, texto, fonte, mini gráfico e sugestões.
4. **Verificação:** todo número da resposta precisa existir nos dados entregues; linguagem de causa é barrada; uma correção automática; senão, resposta segura.

In [13]:
exportar_web.main()
base_conhecimento.main()
dados_agente.main()

web\public\dados.json: 132 KB · web\public\sc_microrregioes.geojson: 708 KB
web\api\_base.js: 21,892 caracteres (~5,473 tokens)
web\api\_dados.js: 194 KB · 628 linhas de recorte · 295 municípios


WindowsPath('C:/Users/user/OneDrive/Área de Trabalho/datathon/datathon/web/api/_dados.js')

## Premissas e limites

1. O CAGED mede **fluxo** (admissões e desligamentos), não vagas abertas nem desemprego: "apagão" é inferido por sinais indiretos.
2. Escolaridade e salário são **associados** às saídas; nenhuma análise aqui afirma causa.
3. Indústria = seções B–F; análise principal na transformação (C); construção separada.
4. Linhas idênticas mantidas (provável contratação em lote), como nos números oficiais.
5. O prêmio de contratação era hipótese da equipe; o modelo mostrou que ele não se sustenta com controles.
6. Limiares (volumes mínimos, cortes) são escolhas da equipe, com testes de sensibilidade em `reports/`.

A lista completa está no README e em `docs/SDD_v2.md`.